In [ ]:
import numpy as np
from parameters import get_parameters, get_slider_params, calculate_derived_parameters
from model_run import run_model_dash
from global_func import reset_flags, reset_E, reset_HSS, reset_S
import math
import matplotlib.pyplot as plt
import pandas as pd
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

In [ ]:
total_runs = 100
seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=total_runs)
MODEL = {
    "int_period": 36,
    "n_months": 36,
}
slider_params = get_slider_params()
results = []
for run in range(total_runs):
    base_seed = seeds[run]
    rng_param = np.random.default_rng(base_seed)
    b_param = get_parameters(rng = rng_param)
    b_param = calculate_derived_parameters(b_param)
    b_flags = reset_flags()
    b_HSS = reset_HSS(slider_params)
    b_S = reset_S(slider_params)
    b_E = reset_E()
    b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS
    })
    n_months = MODEL["n_months"]
    int_period = MODEL["int_period"]
    _, outcomes = run_model_dash(b_param, b_flags, n_months, int_period, base_seed=base_seed)

    outcomes['i_CS'] = np.where(outcomes['i_mod'].isin(["EmCS", "ELCS"]), 1, 0)
    outcomes['i_loc_grouped'] = np.where(outcomes['i_loc_new_v2'] == 0, 0,
                                         np.where(outcomes['i_loc_new_v2'] == 1, 1, 2))
    outcomes['i_facility'] = np.where(outcomes['i_loc_new_v2'] > 0, 1, 0)
    outcomes['i_home'] = np.where(outcomes['i_loc_new_v2'] == 0, 1, 0)
    mcr_l23_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 1]['i_comp_death_new'].mean(), nan=0.0)
    mcr_l45_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 2]['i_comp_death_new'].mean(), nan=0.0)
    p_mcr_l23_l45 = mcr_l23_mean / mcr_l45_mean if mcr_l45_mean != 0 else 0

    results.append({
        #"p_home_pre": (outcomes['i_loc'] == 0).mean(),
        #"p_l23_pre": (outcomes['i_loc'] == 1).mean(),
        #"p_l45_pre": (outcomes['i_loc'] >= 2).mean(),
        #"p_home_anc": outcomes[(outcomes['i_ANC'] == 1)]['i_home'].mean(),
        "p_elcs_fac": outcomes.groupby('i_facility')['i_elec_CS'].mean().get(1, 0),
        "p_elcs_preterm_l45": outcomes[(outcomes['i_loc_grouped'] == 2) & (outcomes['i_preterm'] == 1)]['i_elec_CS'].mean(),
        "p_preterm_l23": outcomes[(outcomes['i_loc_grouped'] == 1)]['i_preterm'].mean(),
        "p_preterm_l45": outcomes[(outcomes['i_loc_grouped'] == 2)]['i_preterm'].mean(),
        "p_cs_l23": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(1, 0),
        "p_cs_l45": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(2, 0),
        "p_unnecessary_cs": outcomes[(outcomes['i_mod'] == "EmCS")]['i_unnecessary_cs'].mean(),

        "p_l23_post": (outcomes['i_loc_grouped'] == 1).mean(),
        "p_l45_post": (outcomes['i_loc_grouped'] == 2).mean(),
        "p_mcr_l23_l45": p_mcr_l23_l45,
        "p_death_SMO": outcomes[(outcomes['i_severe_new'] == 1)]['i_mat_death'].mean(),
        "p_SMO_fac": outcomes[(outcomes['i_severe_new'] == 1) & (outcomes['i_facility'] == 1)]['i_mat_death'].mean(),

        "p_aph_fac": outcomes.groupby('i_facility')['i_aph'].mean().get(1, 0),
        "p_pph_l45": outcomes.groupby('i_loc_grouped')['i_pph_new'].mean().get(2, 0),
        "p_mat_sepsis_l45": outcomes.groupby('i_loc_grouped')['i_mat_sepsis'].mean().get(2, 0),
        "p_eclampsia_fac": outcomes.groupby('i_facility')['i_eclampsia'].mean().get(1, 0),
        "p_obstructed_fac": outcomes.groupby('i_facility')['i_OL_final'].mean().get(1, 0),
        "p_uterine_fac": outcomes.groupby('i_facility')['i_ruptured_uterus'].mean().get(1, 0),

        "MMR_home":  outcomes[(outcomes['i_loc_grouped'] == 0)]['i_mat_death'].mean() * 100000,
        "MMR_l23":  outcomes[(outcomes['i_loc_grouped'] == 1)]['i_mat_death'].mean() * 100000,
        "MMR_l45":  outcomes[(outcomes['i_loc_grouped'] == 2)]['i_mat_death'].mean() * 100000,
        "MMR_aph": ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "aph")).mean() * 100000,
        "MMR_pph": ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "pph")).mean() * 100000,
        "MMR_eclampsia": ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "eclampsia")).mean() * 100000,
        "MMR_ol": ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "ol")).mean() * 100000,
        "MMR_sepsis": ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "sepsis")).mean() * 100000,
        "MMR_other": ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "other")).mean() * 100000,
    })

    # Convert to DataFrame and compute mean
df_results = pd.DataFrame(results).mean().to_dict()
#round 2 digits for each cell of df_results
df_results = {k: round(v, 2) for k, v in df_results.items()}
df_results

In [ ]:
# Define the parameter space for Bayesian Optimization
param_space = [
    Real(0.01, 0.2,      name='p_elec_CS|highrisk'), #control "p_elcs_fac"
    Real(0.5, 0.8,      name='p_elec_CS|preterm'),  #control "p_elcs_preterm_l45"
    Real(80, 100,       name='t_l23_l45_preterm'), #control 'p_preterm_l23' and 'p_preterm_l45'
    Real(0.01, 0.10,    name='p_cs_capacity_l23'), #control "p_cs_l23"
    Real(0.10, 0.20,    name='p_cs_capacity_l45'), #control "p_cs_l45"

    Real(0.5, 0.74,     name='sen_comp_trad'),    #control "p_unnecessary_cs"
    Real(0.5, 0.77,     name='spec_comp_trad'),   #control "p_unnecessary_cs"
    Real(50, 100,       name='t_l23_l45_notsevere'), #control 'p_l23_post' and 'p_l45_post' and "p_mcr_l23_l45"
    Real(50, 100,       name='t_l23_l45_severe'),   #control 'p_l23_post' and 'p_l45_post' and "p_mcr_l23_l45"
    Real(0.01, 0.10,    name='p_comp_severe_lowrisk'), #control "p_SMO_fac"

    Real(0.01, 0.03,    name='p_aph'),
    Real(0.01, 0.03,    name='p_eclampsia'),
    Real(0.01, 0.03,    name='p_ruptured_uterus'),
    Real(0.70, 1.00,    name='p_OL_scale'),
    Real(0.01, 0.02,    name='p_pph_other'),
    Real(0.03, 0.05,    name='p_mat_sepsis_other'),

    Real(0.01, 0.50,         name='p_MM_home'),             #control "MMR_home"
    Real(5, 10,              name='weight_facility_mat'),             #control "MMR_l23" and "MMR_l45"
    Real(0.001, 0.005,       name='p_MM_others'),         #control MMR_other
    Real(0.10, 2.00,         name='MM_weight_pph'),    #control MMR_pph
    Real(0.10, 2.00,         name='MM_weight_sepsis'), #control MMR_sepsis
    Real(0.10, 2.00,         name='MM_weight_eclampsia'), #control MMR_eclampsia
    Real(0.10, 2.00,         name='MM_weight_ol'),      #control MMR_ol
    Real(0.10, 2.00,         name='MM_weight_aph'),    #control MMR_aph
]

# Calibration target values
target_values = {
    "p_elcs_fac": 0.066,
    "p_elcs_preterm_l45": 0.265,
    "p_preterm_l23": 0.035,
    "p_preterm_l45": 0.209,
    "p_cs_l23": 0.024,
    "p_cs_l45": 0.264,
    "p_unnecessary_cs": 0.67,

    "p_l23_post": 0.27,
    "p_l45_post": 0.38,
    "p_mcr_l23_l45": 0.2,
    "p_SMO_fac": 0.0375,

    "p_aph_fac": 0.018,
    "p_pph_l45": 0.086,
    "p_mat_sepsis_l45": 0.15,
    "p_eclampsia_fac": 0.02,
    "p_obstructed_fac": 0.045,
    "p_uterine_fac": 0.0197,

    "p_death_SMO": 0.05,
    "MMR_home": 457,
    "MMR_l23": 27,
    "MMR_l45": 116,
    "MMR_aph": 31.6,
    "MMR_pph": 52.2,
    "MMR_eclampsia": 54.6,
    "MMR_ol": 7.3,
    "MMR_sepsis": 35.0,
    "MMR_other": 30.8,
}

# Generate unique seeds for reproducibility
total_runs = 100
seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=total_runs)

# Objective function for Bayesian Optimization
@use_named_args(param_space)
def objective(**params):

    from parameters import get_parameters, get_slider_params
    from model_run import run_model_dash
    from global_func import reset_flags, reset_E, reset_HSS, reset_S

    MODEL = {
        "int_period": 36,
        "n_months": 36,
    }

    slider_params = get_slider_params()
    results = []

    for run in range(total_runs):
        base_seed = seeds[run]
        rng_param = np.random.default_rng(base_seed)
        b_param = get_parameters(rng=rng_param)
        b_flags = reset_flags()
        b_HSS = reset_HSS(slider_params)
        b_S = reset_S(slider_params)
        b_E = reset_E()
        b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS})

        # Replace parameters with those to calibrate
        b_param['p_elec_CS|highrisk'] = params['p_elec_CS|highrisk']
        b_param['p_elec_CS|preterm'] = params['p_elec_CS|preterm']
        b_param['t_l23_l45_preterm'] = params['t_l23_l45_preterm']
        b_param['p_cs_capacity'][1] = params['p_cs_capacity_l23']
        b_param['p_cs_capacity'][2] = params['p_cs_capacity_l45']
        b_param['p_cs_capacity'][3] = params['p_cs_capacity_l45']
        b_param['sen_comp_trad'] = params['sen_comp_trad']
        b_param['spec_comp_trad'] = params['spec_comp_trad']
        b_param['t_l23_l45_notsevere'] = params['t_l23_l45_notsevere']
        b_param['t_l23_l45_severe'] = params['t_l23_l45_severe']

        b_param['p_aph'] = params['p_aph']
        b_param['p_eclampsia'] = params['p_eclampsia']
        b_param['p_ruptured_uterus'] = params['p_ruptured_uterus']
        b_param['p_OL_scale'] = params['p_OL_scale']
        b_param['p_pph_other'] = params['p_pph_other']
        b_param['p_mat_sepsis_other'] = params['p_mat_sepsis_other']
        b_param['p_comp_severe_lowrisk'] = params['p_comp_severe_lowrisk']

        b_param['p_MM_home'] = params['p_MM_home']
        b_param['weight_facility_mat'] = params['weight_facility_mat']
        b_param['p_MM_others'] = params['p_MM_others']
        b_param['MM_weight_pph'] = params['MM_weight_pph']
        b_param['MM_weight_sepsis'] = params['MM_weight_sepsis']
        b_param['MM_weight_eclampsia'] = params['MM_weight_eclampsia']
        b_param['MM_weight_ol'] = params['MM_weight_ol']
        b_param['MM_weight_aph'] = params['MM_weight_aph']

        #update parameters
        b_param = calculate_derived_parameters(b_param)

        #run the model
        n_months = MODEL["n_months"]
        int_period = MODEL["int_period"]
        _, outcomes = run_model_dash(b_param, b_flags, n_months, int_period, base_seed=base_seed)

        # Calculate outcomes
        outcomes['i_CS'] = np.where(outcomes['i_mod'].isin(["EmCS", "ELCS"]), 1, 0)
        outcomes['i_loc_grouped'] = np.where(outcomes['i_loc_new_v2'] == 0, 0,
                                             np.where(outcomes['i_loc_new_v2'] == 1, 1, 2))
        outcomes['i_facility'] = np.where(outcomes['i_loc_new_v2'] > 0, 1, 0)
        outcomes['i_home'] = np.where(outcomes['i_loc_new_v2'] == 0, 1, 0)
        mcr_l23_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 1]['i_comp_death_new'].mean(), nan=0.0)
        mcr_l45_mean = np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 2]['i_comp_death_new'].mean(), nan=0.0)
        p_mcr_l23_l45 = mcr_l23_mean / mcr_l45_mean if mcr_l45_mean != 0 else 0

        results.append({
            "p_elcs_fac": outcomes.groupby('i_facility')['i_elec_CS'].mean().get(1, 0),
            "p_elcs_preterm_l45": np.nan_to_num(
                outcomes[(outcomes['i_loc_grouped'] == 2) & (outcomes['i_preterm'] == 1)]['i_elec_CS'].mean(), nan=0.0),
            "p_preterm_l23": np.nan_to_num(
                outcomes[(outcomes['i_loc_grouped'] == 1)]['i_preterm'].mean(), nan=0.0),
            "p_preterm_l45": np.nan_to_num(
                outcomes[(outcomes['i_loc_grouped'] == 2)]['i_preterm'].mean(), nan=0.0),
            "p_cs_l23": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(1, 0),
            "p_cs_l45": outcomes.groupby('i_loc_grouped')['i_CS'].mean().get(2, 0),
            "p_unnecessary_cs": np.nan_to_num(
                outcomes[outcomes['i_mod'] == "EmCS"]['i_unnecessary_cs'].mean(), nan=0.0),

            "p_l23_post": np.nan_to_num((outcomes['i_loc_grouped'] == 1).mean(), nan=0.0),
            "p_l45_post": np.nan_to_num((outcomes['i_loc_grouped'] == 2).mean(), nan=0.0),
            "p_mcr_l23_l45": p_mcr_l23_l45,
            "p_death_SMO": np.nan_to_num(
                outcomes[outcomes['i_severe_new'] == 1]['i_mat_death'].mean(), nan=0.0),
            "p_SMO_fac": np.nan_to_num(
                outcomes[(outcomes['i_severe_new'] == 1) & (outcomes['i_facility'] == 1)]['i_mat_death'].mean(), nan=0.0),

            "p_aph_fac": outcomes.groupby('i_facility')['i_aph'].mean().get(1, 0),
            "p_pph_l45": outcomes.groupby('i_loc_grouped')['i_pph_new'].mean().get(2, 0),
            "p_mat_sepsis_l45": outcomes.groupby('i_loc_grouped')['i_mat_sepsis'].mean().get(2, 0),
            "p_eclampsia_fac": outcomes.groupby('i_facility')['i_eclampsia'].mean().get(1, 0),
            "p_obstructed_fac": outcomes.groupby('i_facility')['i_OL_final'].mean().get(1, 0),
            "p_uterine_fac": outcomes.groupby('i_facility')['i_ruptured_uterus'].mean().get(1, 0),

            "MMR_home": np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 0]['i_mat_death'].mean(), nan=0.0) * 100000,
            "MMR_l23": np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 1]['i_mat_death'].mean(), nan=0.0) * 100000,
            "MMR_l45": np.nan_to_num(outcomes[outcomes['i_loc_grouped'] == 2]['i_mat_death'].mean(), nan=0.0) * 100000,

            "MMR_aph": np.nan_to_num(
                ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "aph")).mean(), nan=0.0) * 100000,
            "MMR_pph": np.nan_to_num(
                ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "pph")).mean(), nan=0.0) * 100000,
            "MMR_eclampsia": np.nan_to_num(
                ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "eclampsia")).mean(), nan=0.0) * 100000,
            "MMR_ol": np.nan_to_num(
                ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "ol")).mean(), nan=0.0) * 100000,
            "MMR_sepsis": np.nan_to_num(
                ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "sepsis")).mean(), nan=0.0) * 100000,
            "MMR_other": np.nan_to_num(
                ((outcomes['i_mat_death'] == 1) & (outcomes["death_cause"] == "other")).mean(), nan=0.0) * 100000,
        })


    # Convert to DataFrame and compute mean
    df_results = pd.DataFrame(results).mean().to_dict()

    # Add this line here for early stopping access
    objective.last_df_results = df_results

    # Compute relative RMSE
    relative_squared_errors = [ ((df_results[k] - target_values[k]) / target_values[k])**2 for k in target_values ]
    rmse = math.sqrt(sum(relative_squared_errors) / len(relative_squared_errors))

    # Print current evaluation
    print("\n--- Calibration Iteration ---")
    print("Parameters:")
    for k, v in params.items():
        print(f"  {k:22s}: {v:.4f}")

    print("\nTarget vs Simulated Outcomes:")
    for k in target_values:
        sim = df_results.get(k, 0)
        target = target_values[k]
        error = sim - target
        rel_error = error / target
        print(f"  {k:22s} | Simulated: {sim:.4f}, Target: {target:.4f}, Relative Error: {rel_error:+.2%}")


    print(f"\nRMSE Loss: {rmse:.6f}")
    print("-----------------------------\n")

    return rmse

In [ ]:
# Track RMSE history
rmse_history = []

class EarlyStopper:
    def __init__(self, threshold_rmse=0.05, relative_threshold=0.05):
        self.threshold_rmse = threshold_rmse
        self.relative_threshold = relative_threshold
        self.best_loss = float('inf')
        self.call_count = 0
        self.best_params = None
        self.best_df_results = None

    def meets_relative_criteria(self, sim_results):
        for k in target_values:
            target = target_values[k]
            sim = sim_results.get(k, 0)
            rel_error = abs(sim - target) / target
            if rel_error > self.relative_threshold:
                return False
        return True

    def __call__(self, *args, **kwargs):
        loss = objective(*args, **kwargs)
        rmse_history.append(loss)

        self.call_count += 1
        if loss < self.best_loss:
            self.best_loss = loss
            self.best_params = args[0] if args else None

        sim_results = getattr(objective, "last_df_results", {})
        if self.best_loss < self.threshold_rmse and self.meets_relative_criteria(sim_results):
            print(f"\n⏹️ Early stopping triggered. RMSE: {self.best_loss:.6f}")
            raise StopIteration

        return loss

def save_results(best_params, best_loss, rmse_history, param_space):
    param_names = [dim.name for dim in param_space]
    best_param_dict = dict(zip(param_names, best_params))
    best_param_dict["Best RMSE"] = best_loss
    df_best = pd.DataFrame([best_param_dict])
    df_best.to_csv("best_parameters_whole.csv", index=False)

    pd.DataFrame({"Iteration": range(1, len(rmse_history)+1), "RMSE": rmse_history}).to_csv("rmse_history_whole.csv", index=False)

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(rmse_history)+1), rmse_history, marker='o')
    plt.xlabel("Iteration")
    plt.ylabel("RMSE Loss")
    plt.title("RMSE Loss Over Iterations")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("rmse_plot_whole.png")
    plt.close()

In [ ]:
def optimize_model():
    early_stopper = EarlyStopper(threshold_rmse=0.05, relative_threshold=0.05)
    try:
        result = gp_minimize(
            func=early_stopper,
            dimensions=param_space,
            n_calls=300,
            random_state=42,
            n_random_starts=10,
            verbose=True
        )
        print("\n✅ Optimization completed.")
        print("Best Parameters:", result.x)
        print("Best Loss:", result.fun)
        save_results(result.x, result.fun, rmse_history, param_space)
        return result

    except StopIteration:
        print("\n✅ Optimization stopped early.")
        print(f"Best RMSE achieved: {early_stopper.best_loss:.6f}")
        if early_stopper.best_params is not None:
            print("Best Parameters:")
            for name, val in zip([d.name for d in param_space], early_stopper.best_params):
                print(f"  {name:22s}: {val:.4f}")
            save_results(early_stopper.best_params, early_stopper.best_loss, rmse_history, param_space)
        return None

# Run the optimization
if __name__ == "__main__":
    optimize_model()